이 노트북은 <[케라스 창시자에게 배우는 딥러닝 (개정 3판)](https://tensorflow.blog/kerasdl3/)>(길벗, 2026)의 예제 코드를 담고 있습니다.

### JAX 소개

**준비:** 원본 노트북의 텐서플로 절에서 만든 데이터를 이 노트북에서도 쓰므로 여기서 다시 만듭니다.

In [ ]:
import numpy as np

num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)

In [ ]:
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

In [ ]:
targets = np.vstack(
    (
        np.zeros((num_samples_per_class, 1), dtype="float32"),
        np.ones((num_samples_per_class, 1), dtype="float32"),
    )
)

In [ ]:
x = np.ones(shape=(2, 2))

#### JAX 첫걸음

#### JAX의 텐서

In [56]:
from jax import numpy as jnp
jnp.ones(shape=(2, 1))

Array([[1.],
       [1.]], dtype=float32)

In [57]:
jnp.zeros(shape=(2, 1))

Array([[0.],
       [0.]], dtype=float32)

In [58]:
jnp.array([1, 2, 3], dtype="float32")

Array([1., 2., 3.], dtype=float32)

#### JAX의 난수 생성

In [59]:
np.random.normal(size=(3,))

array([-0.58097322,  1.8175124 ,  0.56813862])

In [60]:
np.random.normal(size=(3,))

array([ 1.01993899,  0.78985267, -0.43526717])

In [ ]:
def apply_noise(x, seed):
    np.random.seed(seed)
    x = x * np.random.normal((3,))
    return x

seed = 1337
y = apply_noise(x, seed)
seed += 1
z = apply_noise(x, seed)

In [ ]:
import jax

seed_key = jax.random.key(1337)

In [63]:
seed_key = jax.random.key(0)
jax.random.normal(seed_key, shape=(3,))

Array([ 1.6226422 ,  2.0252647 , -0.43359444], dtype=float32)

In [64]:
seed_key = jax.random.key(123)
jax.random.normal(seed_key, shape=(3,))

Array([1.6359469 , 0.8408094 , 0.02212393], dtype=float32)

In [65]:
jax.random.normal(seed_key, shape=(3,))

Array([1.6359469 , 0.8408094 , 0.02212393], dtype=float32)

In [66]:
seed_key = jax.random.key(123)
jax.random.normal(seed_key, shape=(3,))

Array([1.6359469 , 0.8408094 , 0.02212393], dtype=float32)

In [67]:
new_seed_key = jax.random.split(seed_key, num=1)[0]
jax.random.normal(new_seed_key, shape=(3,))

Array([-0.49093357, -0.9478693 , -1.775197  ], dtype=float32)

##### 텐서 할당

In [ ]:
x = jnp.array([1, 2, 3], dtype="float32")
new_x = x.at[0].set(10)

##### 텐서 연산: JAX로 수학 계산하기

In [ ]:
a = jnp.ones((2, 2))
b = jnp.square(a)
c = jnp.sqrt(a)
d = b + c
e = jnp.matmul(a, b)
e *= d

In [ ]:
def dense(inputs, W, b):
    return jax.nn.relu(jnp.matmul(inputs, W) + b)

##### JAX로 그레이디언트 계산하기

In [ ]:
def compute_loss(input_var):
    return jnp.square(input_var)

In [ ]:
grad_fn = jax.grad(compute_loss)

In [ ]:
input_var = jnp.array(3.0)
grad_of_loss_wrt_input_var = grad_fn(input_var)

##### JAX 그레이디언트 계산의 모범 사례

###### 손실 값 반환

In [ ]:
grad_fn = jax.value_and_grad(compute_loss)
output, grad_of_loss_wrt_input_var = grad_fn(input_var)

###### 복잡한 함수에 대한 그레이디언트 구하기

###### 보조 출력 반환

##### @jax.jit로 JAX 함수 속도 높이기

In [ ]:
@jax.jit
def dense(inputs, W, b):
    return jax.nn.relu(jnp.matmul(inputs, W) + b)

#### 엔드-투-엔드 예제: 순수 JAX로 구현하는 선형 분류기

In [ ]:
def model(inputs, W, b):
    return jnp.matmul(inputs, W) + b

def mean_squared_error(targets, predictions):
    per_sample_losses = jnp.square(targets - predictions)
    return jnp.mean(per_sample_losses)

In [ ]:
def compute_loss(state, inputs, targets):
    W, b = state
    predictions = model(inputs, W, b)
    loss = mean_squared_error(targets, predictions)
    return loss

In [ ]:
grad_fn = jax.value_and_grad(compute_loss)

In [ ]:
learning_rate = 0.1

@jax.jit
def training_step(inputs, targets, W, b):
    loss, grads = grad_fn((W, b), inputs, targets)
    grad_wrt_W, grad_wrt_b = grads
    W = W - grad_wrt_W * learning_rate
    b = b - grad_wrt_b * learning_rate
    return loss, W, b

In [80]:
input_dim = 2
output_dim = 1

W = jax.numpy.array(np.random.uniform(size=(input_dim, output_dim)))
b = jax.numpy.array(np.zeros(shape=(output_dim,)))
state = (W, b)
for step in range(40):
    loss, W, b = training_step(inputs, targets, W, b)
    print(f"{step}번째 스텝의 손실: {loss:.4f}")

0번째 스텝의 손실: 3.3401
1번째 스텝의 손실: 0.2061
2번째 스텝의 손실: 0.1196
3번째 스텝의 손실: 0.1004
4번째 스텝의 손실: 0.0923
5번째 스텝의 손실: 0.0863
6번째 스텝의 손실: 0.0810
7번째 스텝의 손실: 0.0763
8번째 스텝의 손실: 0.0719
9번째 스텝의 손실: 0.0680
10번째 스텝의 손실: 0.0643
11번째 스텝의 손실: 0.0610
12번째 스텝의 손실: 0.0580
13번째 스텝의 손실: 0.0552
14번째 스텝의 손실: 0.0526
15번째 스텝의 손실: 0.0503
16번째 스텝의 손실: 0.0482
17번째 스텝의 손실: 0.0462
18번째 스텝의 손실: 0.0444
19번째 스텝의 손실: 0.0428
20번째 스텝의 손실: 0.0413
21번째 스텝의 손실: 0.0399
22번째 스텝의 손실: 0.0387
23번째 스텝의 손실: 0.0375
24번째 스텝의 손실: 0.0365
25번째 스텝의 손실: 0.0355
26번째 스텝의 손실: 0.0346
27번째 스텝의 손실: 0.0338
28번째 스텝의 손실: 0.0331
29번째 스텝의 손실: 0.0324
30번째 스텝의 손실: 0.0318
31번째 스텝의 손실: 0.0312
32번째 스텝의 손실: 0.0307
33번째 스텝의 손실: 0.0303
34번째 스텝의 손실: 0.0298
35번째 스텝의 손실: 0.0294
36번째 스텝의 손실: 0.0291
37번째 스텝의 손실: 0.0287
38번째 스텝의 손실: 0.0284
39번째 스텝의 손실: 0.0282


#### JAX 방식의 차이점